In [1]:
# Cell 1: Set working directory and clone the project repo from GitHub
%cd /kaggle/working/
!rm -rf /kaggle/working/Predictive-Analytics-Project
!git clone https://github.com/tom666d/Predictive-Analytics-Project.git

/kaggle/working
Cloning into 'Predictive-Analytics-Project'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 169 (delta 72), reused 110 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 468.17 KiB | 10.64 MiB/s, done.
Resolving deltas: 100% (72/72), done.


In [2]:
# Cell 2: Write the model configuration YAML file
# This config defines:
#   - data: input path, number of training days (n_days=500), validation window (val_days=28)
#   - features: lag features (28/35/42/364/365 days), rolling mean (28 days), and a full list of feature columns
#   - model: XGBoost with Tweedie objective, 2000 estimators, GPU (cuda) training
#   - output: submission CSV path, model save directory, skip_training flag
import os
yaml_content = """
data:
  path: "/kaggle/input/competitions/m5-forecasting-accuracy/"
  n_days: 500
  val_days: 28

features:
  lags: [28, 35, 42, 364, 365]
  rolling_means: [28]
  use_rolling_std: false
  use:
    - item_id
    - store_id
    - state_id
    - dept_id
    - cat_id
    - rmean_28
    - lag_28
    - lag_35
    - lag_42
    - lag_364
    - lag_365
    - weekofyear
    - dayofweek
    - wday
    - month
    - is_weekend
    - sell_price
    - price_mean_7
    - price_change
    - price_vs_mean
    - event_type
    - is_high_impact_event
    - snap

model:
  active_model: "xgb"
  xgb:
    objective: "reg:tweedie"
    tweedie_variance_power: 1.1
    n_estimators: 2000
    learning_rate: 0.02
    max_depth: 7
    subsample: 0.8
    colsample_bytree: 0.8
    min_child_weight: 30
    reg_alpha: 0.3
    reg_lambda: 1.0
    max_bin: 256
    tree_method: "hist"
    device: "cuda"
    early_stopping_rounds: 100

output:
  submission_path: "submission_FL_xgb.csv"
  models_dir: "models/"
  skip_training: false
"""
os.makedirs('/kaggle/working/Predictive-Analytics-Project/configs', exist_ok=True)
with open('/kaggle/working/Predictive-Analytics-Project/configs/config_FL.yaml', 'w') as f:
    f.write(yaml_content)
print("✅ Cell 2: Round 1 zero-risk YAML has been updated.")

✅ Cell 2: Round 1 zero-risk YAML has been updated.


In [3]:
# Cell 3: Write preprocessing.py to the project source directory
# Contains two functions:
#   - reduce_mem(df): Downcasts numeric columns to the smallest fitting dtype
#                     (int8/int16/int32 for integers, float32 for floats) to save RAM
#   - load_and_preprocess(cfg): Loads sales, calendar, and sell_prices CSVs;
#                               melts wide sales data to long format;
#                               merges calendar info (date, wday, month, events, SNAP flags);
#                               merges sell prices; derives a unified snap column per state;
#                               casts categorical columns; sorts by store/item/date
content = """
import os
import numpy as np
import pandas as pd

def reduce_mem(df):
    # Downcast numeric columns to smallest fitting dtype to reduce memory usage
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type) != 'category' and 'datetime' not in str(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                # Try int8 first, then int16, fall back to int32
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                else:
                    df[col] = df[col].astype(np.int32)
            else:
                # Cast all float columns to float32
                df[col] = df[col].astype(np.float32)
    return df

def load_and_preprocess(cfg: dict):
    data_path = cfg["data"]["path"]
    n_days    = cfg["data"]["n_days"]
    # Load calendar and sell prices reference tables
    calendar    = pd.read_csv(os.path.join(data_path, "calendar.csv"))
    sell_prices = pd.read_csv(os.path.join(data_path, "sell_prices.csv"))
    calendar["date"] = pd.to_datetime(calendar["date"])
    # Define ID columns and select only the required day columns based on n_days
    id_cols  = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    start_day = 1913 - (n_days + 28) + 1
    day_cols = [f"d_{i}" for i in range(max(1, start_day), 1914)]
    # Load sales data with only the relevant day columns to save memory
    sales = pd.read_csv(os.path.join(data_path, "sales_train_validation.csv"), usecols=id_cols + day_cols)
    # Melt from wide format (one column per day) to long format (one row per item-day)
    df = sales.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales")
    del sales
    # Merge calendar features: date, weekday info, event info, and state SNAP flags
    cal_cols = ["d", "date", "wm_yr_wk", "weekday", "wday", "month", "year", "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"]
    df = df.merge(calendar[cal_cols], on="d", how="left")
    df["date"] = pd.to_datetime(df["date"])
    # Merge sell prices on store, item, and week
    df = df.merge(sell_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    # Derive a unified snap column: assign each row its state-specific SNAP value
    df["snap"] = 0
    df.loc[df["state_id"] == "CA", "snap"] = df["snap_CA"]
    df.loc[df["state_id"] == "TX", "snap"] = df["snap_TX"]
    df.loc[df["state_id"] == "WI", "snap"] = df["snap_WI"]
    # Unify event type column, fill missing as 'None'
    df["event_type"] = df["event_type_1"].fillna("None")
    # Drop redundant/intermediate columns no longer needed
    df.drop(columns=["snap_CA", "snap_TX", "snap_WI", "event_type_1", "d", "wm_yr_wk", "weekday", "year"], inplace=True)
    # Reduce memory footprint
    df = reduce_mem(df)
    # Cast high-cardinality string columns to category dtype for XGBoost compatibility
    cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id", "event_type"]
    for col in cat_cols: df[col] = df[col].astype("category")
    return df.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True), calendar, sell_prices
"""
with open('/kaggle/working/Predictive-Analytics-Project/src/preprocessing.py', 'w') as f: 
    f.write(content)
print("✅ Cell 3: preprocessing.py is ready.")

✅ Cell 3: preprocessing.py is ready.


In [4]:
# Cell 4: Write models.py to the project source directory
# Contains the following functions:
#   - get_xgb_model(cfg): Instantiates an XGBRegressor from config parameters
#   - train_store_models(...): Trains one XGB model per store, optionally with early stopping
#   - train_ensemble_models(...): Stub — raises NotImplementedError (ensemble not active)
#   - predict_store_ensemble(...): Stub — raises NotImplementedError (ensemble not active)
#   - predict_store(...): Runs per-store prediction and assembles into a single array
content = """
import numpy as np
import xgboost as xgb

def get_xgb_model(cfg: dict):
    # Build an XGBRegressor from the 'xgb' section of the config dict
    p = cfg["model"]["xgb"]
    return xgb.XGBRegressor(
        objective=p["objective"], n_estimators=p["n_estimators"], learning_rate=p["learning_rate"],
        max_depth=p["max_depth"], subsample=p["subsample"], colsample_bytree=p["colsample_bytree"],
        min_child_weight=p.get("min_child_weight", 1), 
        reg_alpha=p.get("reg_alpha", 0), reg_lambda=p.get("reg_lambda", 1),
        max_bin=p.get("max_bin", 256),
        tree_method=p["tree_method"],
        device=p.get("device", "cpu"), enable_categorical=True
    )

def train_store_models(train_df, val_df, features, target, cat_cols, cfg, model_name="xgb"):
    # Train one model per store; use early stopping if val_df is provided
    model_cfg = cfg["model"][model_name]
    early_stop = model_cfg.get("early_stopping_rounds")
    stores = train_df["store_id"].unique()
    models = {}
    for store in stores:
        print(f"  Training {model_name} — store: {store}")
        tr = train_df[train_df["store_id"] == store]
        model = get_xgb_model(cfg)
        if val_df is not None and early_stop:
            # Use validation set for early stopping to prevent overfitting
            va = val_df[val_df["store_id"] == store]
            model.set_params(early_stopping_rounds=early_stop)
            model.fit(tr[features], tr[target], eval_set=[(va[features], va[target])], verbose=False)
        else:
            # Train without validation (e.g., final retraining on full data)
            model.fit(tr[features], tr[target])
        models[store] = model
    return models

# 🛠️ [Stub] Ensemble functions required by train.py imports — not active in this config
def train_ensemble_models(train_df, val_df, features, target, cat_cols, cfg):
    raise NotImplementedError("Ensemble mode is not active in this config.")

def predict_store_ensemble(X, all_models, cfg):
    raise NotImplementedError("Ensemble prediction is not active in this config.")

def predict_store(X, models, store_col="store_id", features=None):
    # Run inference for each store separately and collect predictions
    y_pred = np.zeros(len(X))
    for store in X[store_col].unique():
        if store in models:
            mask = X[store_col] == store
            # If features list is given, slice only those columns; otherwise use full X
            X_input = X.loc[mask, features] if features else X.loc[mask]
            y_pred[mask] = models[store].predict(X_input)
    return y_pred
"""
with open('/kaggle/working/Predictive-Analytics-Project/src/models.py', 'w') as f: 
    f.write(content)
print("✅ Cell 4: models.py with all functions is ready.")

✅ Cell 4: models.py with all functions is ready.


In [5]:
# Cell 5: Patch train.py and launch the full XGBoost training pipeline
import os

# Step 1: Read the existing train.py for patching
with open('/kaggle/working/Predictive-Analytics-Project/src/train.py', 'r') as f:
    train_code = f.read()

# [Bug Fix] Routing fix: ensure store_id is included in feature slice during prediction
# Without store_id, predict_store() cannot route rows to the correct per-store model
if 'X    = full_df.loc[mask, list(set(features + ["store_id"]))]' not in train_code:
    train_code = train_code.replace(
        'X    = full_df.loc[mask, features]',
        'X    = full_df.loc[mask, list(set(features + ["store_id"]))]'
    )
if 'y_pred = predict_store(X, models_or_all, features=features)' not in train_code:
    train_code = train_code.replace(
        'y_pred = predict_store(X, models_or_all)',
        'y_pred = predict_store(X, models_or_all, features=features)'
    )

# [Optimization] 380-day history filter: trim data to the last 380 days before prediction
# This reduces memory pressure and stabilizes recursive forecasting (addresses score ~2.78 issue)
optimization_code = """
    print("\\n── [System] Applying 380-day history filter for stability... ──")
    max_date = df['date'].max()
    cutoff_date = max_date - pd.Timedelta(days=380)
    df = df[df['date'] >= cutoff_date].reset_index(drop=True)
    print(f"── [System] Data shrunk to {len(df)} rows. ──")
    print("\\n── Step 6: Recursive 28-day prediction ──")
"""
if '[System] Applying 380-day' not in train_code:
    train_code = train_code.replace(
        'print("\\n── Step 6: Recursive 28-day prediction ──")',
        optimization_code
    )

# Write the patched train.py back to disk
with open('/kaggle/working/Predictive-Analytics-Project/src/train.py', 'w') as f:
    f.write(train_code)

# Step 2: Run the training pipeline
%cd /kaggle/working/Predictive-Analytics-Project
print("🏁 Starting the optimized XGBoost Pipeline...")
!export PYTHONPATH=/kaggle/working/Predictive-Analytics-Project/src:$PYTHONPATH && python src/train.py --config configs/config_FL.yaml

/kaggle/working/Predictive-Analytics-Project
🏁 Starting the optimized XGBoost Pipeline...
main() started

── Step 1: Preprocessing ──

── Step 2: Feature engineering ──

── Step 3: Train/val split ──
Train: 2014-11-14 → 2016-03-27 (15,245,000 rows)
Val  : 2016-03-28 → 2016-04-24 (853,720 rows)
Features (23): ['item_id', 'store_id', 'state_id', 'dept_id', 'cat_id', 'rmean_28', 'lag_28', 'lag_35', 'lag_42', 'lag_364', 'lag_365', 'weekofyear', 'dayofweek', 'wday', 'month', 'is_weekend', 'sell_price', 'price_mean_7', 'price_change', 'price_vs_mean', 'event_type', 'is_high_impact_event', 'snap']
Categorical: ['item_id', 'store_id', 'state_id', 'dept_id', 'cat_id', 'event_type']

── Step 4: Training (xgb) with validation ──
  Training xgb — store: CA_1
  Training xgb — store: CA_2
  Training xgb — store: CA_3
  Training xgb — store: CA_4
  Training xgb — store: TX_1
  Training xgb — store: TX_2
  Training xgb — store: TX_3
  Training xgb — store: WI_1
  Training xgb — store: WI_2
  Training 